# PyTorch 기본기 연습: torch · nn · optim

학습 루프(`train_practice.py`)를 짜기 전에 필요한 부품을 하나씩 손에 익히는 노트북입니다.

**사용법**
- `None` 또는 `# TODO`로 표시된 곳을 채우고 셀을 실행하세요.
- 각 셀 아래 `check(...)`가 자동 채점합니다. ✅면 통과, ❌면 힌트가 나옵니다.
- **🤔 생각해보기**는 코드를 고치지 말고 결과를 먼저 예측해본 뒤 실행하세요.
- 데이터 다운로드 없이 랜덤 텐서만 씁니다. CPU로도 몇 초면 끝나요.

| 파트 | 내용 |
|---|---|
| 1. torch | 텐서 생성, shape 다루기, argmax/정확도, device, autograd |
| 2. torch.nn | Linear, Conv2d shape 계산, nn.Module 직접 만들기, CrossEntropyLoss, train/eval 모드 |
| 3. torch.optim | SGD로 최솟값 찾기, zero_grad 실험, 선형회귀, 미니 학습 루프 |

In [ ]:
import math
import torch
from torch import nn

torch.manual_seed(0)

def check(fn, hint=""):
    """fn()이 True면 통과. 에러가 나도 멈추지 않고 이유를 보여준다."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok = False
        hint = f"{hint}  (에러 → {type(e).__name__}: {e})"
    print("✅ 통과" if ok else f"❌ 아직: {hint}")

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

---
# 1. torch: 텐서 다루기

## 1-1. 텐서 만들기
- `a`: 모든 값이 0인 2×3 실수 텐서
- `b`: 파이썬 리스트 `[[1, 2], [3, 4]]`로 만든 텐서
- `c`: 표준정규분포 랜덤값, shape `(4, 3, 32, 32)` (CIFAR 이미지 4장짜리 배치 모양)

In [ ]:
a = None  # TODO
b = None  # TODO
c = None  # TODO

check(lambda: a.shape == (2, 3) and a.sum().item() == 0 and a.is_floating_point(), "a는 2×3 실수 영행렬")
check(lambda: b.tolist() == [[1, 2], [3, 4]], "b는 리스트로부터 만든 텐서")
check(lambda: c.shape == (4, 3, 32, 32), "c의 shape은 (4, 3, 32, 32)")

🤔 **생각해보기**: `b.dtype`과 `a.dtype`은 각각 무엇일까요? 예측한 뒤 아래 셀로 확인하세요. `CrossEntropyLoss`의 정답 라벨이 왜 `int64`(long)여야 하는지와 연결해서 생각해보세요.

In [ ]:
print(a.dtype, b.dtype)

## 1-2. shape 바꾸기
PIL이나 numpy 이미지는 보통 `(N, H, W, C)` 순서인데, PyTorch의 Conv2d는 `(N, C, H, W)`를 기대합니다.
- `x1`: `torch.arange(24)`를 shape `(2, 3, 4)`로
- `imgs_chw`: `imgs`(N,H,W,C)를 `(N, C, H, W)`로 (축 순서 바꾸기)
- `flat`: `imgs_chw`를 배치 차원은 유지하고 나머지를 한 줄로 → `(8, 3072)`. 숫자 3072를 직접 쓰지 말고 해보세요.

In [ ]:
x = torch.arange(24)
imgs = torch.randn(8, 32, 32, 3)   # (N, H, W, C)

x1 = None        # TODO
imgs_chw = None  # TODO
flat = None      # TODO

check(lambda: x1.shape == (2, 3, 4) and x1[1, 2, 3].item() == 23, "x1의 shape은 (2, 3, 4)")
check(lambda: imgs_chw.shape == (8, 3, 32, 32) and imgs_chw[0, 1, 5, 7] == imgs[0, 5, 7, 1],
      "축을 '재배열'해야 함. reshape는 데이터 순서를 섞지 않고 모양만 바꾼다")
check(lambda: flat.shape == (8, 3072), "앞 차원은 그대로 두고, 나머지는 -1로 맡기기")

🤔 **생각해보기**: `imgs.reshape(8, 3, 32, 32)`도 shape은 똑같이 나옵니다. 그런데 왜 틀린 걸까요?

## 1-3. 예측 클래스와 정확도
모델 출력(logits) 4개 샘플, 클래스 3개입니다.
- `preds`: 각 샘플에서 가장 큰 값의 클래스 번호
- `n_correct`: 맞힌 개수 (**파이썬 int**)
- `acc`: 정확도 (**파이썬 float**)

In [ ]:
logits = torch.tensor([[ 2.0, 0.1, -1.0],
                       [ 0.5, 0.2,  3.0],
                       [ 1.0, 4.0,  0.0],
                       [ 0.3, 0.2,  0.1]])
labels = torch.tensor([0, 2, 0, 0])

preds = None      # TODO
n_correct = None  # TODO
acc = None        # TODO

check(lambda: preds.tolist() == [0, 2, 1, 0], "어느 차원을 따라 최댓값 위치를 찾아야 할까? (샘플 방향 vs 클래스 방향)")
check(lambda: type(n_correct) is int and n_correct == 3, "텐서를 파이썬 숫자로 바꾸는 메서드가 필요")
check(lambda: type(acc) is float and abs(acc - 0.75) < 1e-6, "acc는 파이썬 float")

## 1-4. device 옮기기
- `t_dev`: `t`를 위에서 정한 `device`로 옮긴 것
- `t_back`: `t_dev`를 다시 CPU로 옮긴 것

In [ ]:
t = torch.ones(3)

t_dev = None   # TODO
t_back = None  # TODO

check(lambda: t_dev.device.type == device.type, "t_dev는 device 위에 있어야 함")
check(lambda: t_back.device.type == "cpu", "t_back은 cpu 위에 있어야 함")
print("원본 t는 여전히:", t.device, "← .to()는 제자리 변경일까, 새 텐서를 반환할까?")

## 1-5. autograd: 기울기 계산
$y = \sum x_i^2$ 일 때 $\partial y / \partial x_i = 2x_i$ 입니다.
- `y`를 계산하고, 역전파를 호출해서 `x.grad`가 채워지게 하세요.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

y = None  # TODO
# TODO: 역전파 한 줄

check(lambda: x.grad.tolist() == [2.0, 4.0, 6.0], "y를 만든 뒤 y에서 역전파를 호출했는지 확인")

## 1-6. 기울기는 누적된다
🤔 **생각해보기**: 아래 셀은 같은 계산을 **한 번 더** 하고 역전파합니다. 실행 전에 `x.grad`가 어떻게 될지 예측해보세요.

In [ ]:
y2 = (x ** 2).sum()
y2.backward()
print(x.grad)

이제 `x.grad`를 다시 0으로(또는 None으로) 만들어 보세요. 학습 루프에서 `optimizer.zero_grad()`가 하는 일이 바로 이것입니다.

In [ ]:
# TODO: x의 기울기 초기화

check(lambda: x.grad is None or x.grad.abs().sum().item() == 0, "x.grad를 0 또는 None으로")

## 1-7. 기울기 끄기
평가할 때는 기울기 기록이 필요 없습니다. 아래 `z = x * 2`가 기울기를 추적하지 **않도록** 감싸 보세요.

In [ ]:
# TODO: 아래 한 줄을 감싸기
z = x * 2

check(lambda: z.requires_grad is False, "기울기 계산을 끄는 컨텍스트 안에서 z를 만들어야 함")

---
# 2. torch.nn: 층과 모델

## 2-1. nn.Linear
- `fc`: 입력 4차원 → 출력 3차원 선형층
- `w_shape`: `fc.weight`의 shape을 **코드 실행 전에 예측해서** 튜플로 적기 (예: `(?, ?)`)
- `out`: `fc`에 `inp`(배치 5개)를 넣은 결과

In [ ]:
inp = torch.randn(5, 4)

fc = None       # TODO
w_shape = None  # TODO: 예측값을 직접 적기
out = None      # TODO

check(lambda: tuple(fc.weight.shape) == w_shape, f"실제 weight shape은 {tuple(fc.weight.shape) if fc is not None else '?'}. 왜 그렇게 저장할까?")
check(lambda: out.shape == (5, 3), "out은 (5, 3)")

## 2-2. Conv2d / MaxPool2d 출력 shape 계산
출력 크기 공식: $\text{out} = \lfloor (\text{in} + 2p - k) / s \rfloor + 1$

- `conv`: 입력 채널 3 → 출력 채널 16, kernel 3, padding 1
- `pool`: 2×2 max pooling
- 각 단계의 출력 shape을 **먼저 손으로 계산해서** 튜플로 적은 뒤 실행하세요.

In [ ]:
inp = torch.randn(8, 3, 32, 32)

conv = None  # TODO
pool = None  # TODO
conv5 = nn.Conv2d(3, 16, kernel_size=5, padding=0)  # 비교용 (이미 만들어 둠)

shape_after_conv = None   # TODO: 예측 (N, C, H, W)
shape_after_pool = None   # TODO: 예측
shape_after_conv5 = None  # TODO: 예측

check(lambda: conv(inp).shape == shape_after_conv, "padding=1, kernel=3이면 H, W는?")
check(lambda: pool(conv(inp)).shape == shape_after_pool, "2×2 pooling은 H, W를 어떻게 바꿀까?")
check(lambda: conv5(inp).shape == shape_after_conv5, "padding 없이 kernel=5면 32 → ?")

## 2-3. nn.Module 직접 만들기
아래 구조의 `TinyCNN`을 완성하세요.

```
입력 (N, 3, 32, 32)
→ Conv2d(3→16, k=3, p=1) → ReLU → MaxPool(2)
→ Conv2d(16→32, k=3, p=1) → ReLU → MaxPool(2)
→ flatten
→ Linear(?, num_classes)
```
- `?`에 들어갈 숫자를 직접 계산해야 합니다.
- `n_params`: 학습되는 파라미터 총개수를 **코드로** 계산하세요.

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: 층 정의

    def forward(self, x):
        # TODO: 순전파
        return None


model = TinyCNN()
out = model(torch.randn(4, 3, 32, 32))
n_params = None  # TODO

check(lambda: out.shape == (4, 10), "출력은 (배치, 클래스수) = (4, 10)")
check(lambda: n_params == 25578,
      "파라미터 수가 다름. 구조가 위 그림과 똑같은지, 개수를 세는 코드가 맞는지 확인 "
      "(힌트: model.parameters()의 각 텐서 원소 수를 더하기)")

🤔 **생각해보기**: 25,578개 중 가장 많은 파라미터를 가진 층은 어디일까요? 원본 프로젝트의 `gap` 모델이 마지막에 Global Average Pooling을 쓰는 이유와 연결해 보세요.

## 2-4. CrossEntropyLoss
- `criterion`: 분류용 loss
- `loss_uniform`: 모든 logit이 0인 출력(=모든 클래스 확률이 같음)에 대한 loss → 값이 $\ln 10 \approx 2.30$ 이어야 합니다.

In [ ]:
labels = torch.randint(0, 10, (4,))
logits_zero = torch.zeros(4, 10)

criterion = None     # TODO
loss_uniform = None  # TODO

check(lambda: abs(loss_uniform.item() - math.log(10)) < 1e-4, "모든 클래스 확률이 1/10이면 loss는 -ln(1/10)")

### 흔한 실수: softmax를 두 번 하기
정답 클래스에 아주 큰 logit(10.0)을 준 '거의 완벽한' 출력입니다.
- `loss_right`: logits를 그대로 넣은 loss
- `loss_wrong`: logits에 softmax를 **한 번 더** 적용해서 넣은 loss

In [ ]:
logits_conf = torch.zeros(4, 10)
logits_conf[torch.arange(4), labels] = 10.0

loss_right = None  # TODO
loss_wrong = None  # TODO

print("loss_right:", loss_right, "\nloss_wrong:", loss_wrong)
check(lambda: loss_right.item() < 0.01, "거의 완벽한 예측이니 loss는 0에 가까워야 함")
check(lambda: loss_wrong.item() > 1.0, "softmax(dim=1)을 먼저 적용한 뒤 criterion에 넣기")

🤔 **생각해보기**: 예측이 거의 완벽한데도 `loss_wrong`은 왜 크게 나올까요? softmax 출력값(0~1)이 다시 logit으로 쓰이면 무슨 일이 생기는지 생각해보세요.

## 2-5. train 모드 vs eval 모드
Dropout은 모드에 따라 다르게 동작합니다.
- `out_train`: `drop`을 **train 모드**로 두고 `ones`를 통과시킨 결과
- `out_eval`: `drop`을 **eval 모드**로 두고 통과시킨 결과

In [ ]:
torch.manual_seed(1)
drop = nn.Dropout(p=0.5)
ones = torch.ones(1, 10)

# TODO: train 모드로 전환
out_train = None  # TODO
# TODO: eval 모드로 전환
out_eval = None   # TODO

print("train:", out_train, "\neval: ", out_eval)
check(lambda: set(out_train.unique().tolist()) <= {0.0, 2.0} and (out_train == 0).any(), "train 모드에서는 일부가 0이 되어야 함")
check(lambda: torch.equal(out_eval, ones), "eval 모드에서는 입력 그대로")

🤔 **생각해보기**: train 모드에서 살아남은 값은 왜 1이 아니라 **2.0**일까요? 그리고 평가 함수에서 `model.eval()`을 빼먹으면 BN/Dropout 모델의 val acc는 어떻게 될까요?

---
# 3. torch.optim: 파라미터 업데이트

## 3-1. SGD로 최솟값 찾기
$f(w) = (w - 3)^2$ 의 최솟값 위치(w=3)를 SGD로 찾아봅니다.
- `optimizer`: `w`를 업데이트하는 SGD, lr=0.1
- 100번 반복: loss 계산 → 기울기 초기화 → 역전파 → 업데이트 (순서는 직접 판단)

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

optimizer = None  # TODO

for step in range(100):
    # TODO
    pass

print("w =", w.item())
check(lambda: abs(w.item() - 3) < 1e-3, "w가 3에 가까워야 함. 반복문 안에 필요한 4가지가 모두 있는지 확인")

## 3-2. 실험: zero_grad를 빼먹으면?
🤔 3-1의 코드를 아래에 복사하고 **기울기 초기화 줄만 지운 뒤** 실행하세요. 매 10 step마다 `w`를 출력해보면 더 잘 보입니다. 실행 전에 결과를 먼저 예측해보세요.

In [ ]:
# TODO: 3-1 코드 복사 후 zero_grad만 제거해서 실행

## 3-3. 선형회귀: nn + optim 같이 쓰기
`y = 2x + 1`(+약간의 노이즈) 데이터에서 기울기 2와 절편 1을 학습으로 찾아보세요.
- `model`: 입력 1 → 출력 1 선형층
- `criterion`: 평균제곱오차 loss
- `optimizer`: SGD, lr=0.1
- 전체 데이터로 200번 반복 학습

In [ ]:
torch.manual_seed(0)
X = torch.randn(100, 1)
Y = 2 * X + 1 + 0.01 * torch.randn(100, 1)

model = None      # TODO
criterion = None  # TODO
optimizer = None  # TODO

for epoch in range(200):
    # TODO
    pass

print("weight:", model.weight.item(), " bias:", model.bias.item())
check(lambda: abs(model.weight.item() - 2) < 0.05 and abs(model.bias.item() - 1) < 0.05,
      "weight≈2, bias≈1 이어야 함")

## 3-4. 미니 학습 루프: 작은 배치 과적합 테스트
지금까지 배운 걸 전부 합칩니다. 랜덤 이미지 64장에 랜덤 라벨을 붙이고, 2-3의 `TinyCNN`이 이 64장을 **외워버리게** 만드세요. 원본 프로젝트의 `sanity_check.py`가 하는 '과적합 테스트'와 같은 아이디어입니다. 모델이 작은 데이터조차 외우지 못하면 학습 코드 어딘가가 잘못된 거예요.

- 모델, loss, optimizer(Adam, lr=1e-3)를 만들고 `device`로 옮기기
- 같은 64장으로 500 step 학습 (100 step마다 loss 출력)
- 마지막에 **평가 모드**에서 정확도 `final_acc`(파이썬 float) 계산

In [ ]:
torch.manual_seed(0)
imgs = torch.randn(64, 3, 32, 32)
targets = torch.randint(0, 10, (64,))

# TODO: 준비 (model, criterion, optimizer, device 이동)

# TODO: 학습 루프

# TODO: 평가
final_acc = None

print("final_acc:", final_acc)
check(lambda: type(final_acc) is float and final_acc >= 0.95,
      "64장을 거의 다 맞혀야 함. loss가 안 줄면: zero_grad 위치, model.train(), lr 확인")

---
## 다음 단계
여기까지 ✅가 모두 나왔다면, `train_practice.py`의 TODO 1~5는 지금 쓴 코드를
**DataLoader로 배치를 돌면서** 반복하는 것뿐입니다. 3-4와 무엇이 다른지
(배치 단위 누적, epoch 반복, val 평가) 생각하면서 넘어가 보세요.